In [1]:
import numpy as np
from testCases import *
from public_tests import *
from gc_utils import sigmoid, relu, dictionary_to_vector, vector_to_dictionary, gradients_to_vector

%load_ext autoreload
%autoreload 2

In [ ]:
# GRADED FUNCTION: forward_propagation


def forward_propagation(x, theta):
    """
    Implement the linear forward propagation (compute J) presented in Figure 1 (J(theta) = theta * x)

    Arguments:
    x -- a real-valued input
    theta -- our parameter, a real number as well

    Returns:
    J -- the value of function J, computed using the formula J(theta) = theta * x
    """

    # (approx. 1 line)
    # J =
    # YOUR CODE STARTS HERE
    J = theta * x
    # YOUR CODE ENDS HERE

    return J

In [ ]:
x, theta = 2, 4
J = forward_propagation(x, theta)
print("J = " + str(J))
forward_propagation_test(forward_propagation)

J = 8
 All tests passed.


In [ ]:
# GRADED FUNCTION: backward_propagation


def backward_propagation(x, theta):
    """
    Computes the derivative of J with respect to theta (see Figure 1).

    Arguments:
    x -- a real-valued input
    theta -- our parameter, a real number as well

    Returns:
    dtheta -- the gradient of the cost with respect to theta
    """

    # (approx. 1 line)
    # dtheta =
    # YOUR CODE STARTS HERE
    dtheta = x
    # YOUR CODE ENDS HERE

    return dtheta

In [ ]:
x, theta = 3, 4
dtheta = backward_propagation(x, theta)
print("dtheta = " + str(dtheta))
backward_propagation_test(backward_propagation)

dtheta = 3
 All tests passed.


In [ ]:
# GRADED FUNCTION: gradient_check


def gradient_check(x, theta, epsilon=1e-7, print_msg=False):
    """
    Implement the gradient checking presented in Figure 1.

    Arguments:
    x -- a float input
    theta -- our parameter, a float as well
    epsilon -- tiny shift to the input to compute approximated gradient with formula(1)

    Returns:
    difference -- difference (2) between the approximated gradient and the backward propagation gradient. Float output
    """

    # Compute gradapprox using right side of formula (1). epsilon is small enough, you don't need to worry about the limit.
    # (approx. 5 lines)
    # theta_plus =                                 # Step 1
    # theta_minus =                                # Step 2
    # J_plus =                                    # Step 3
    # J_minus =                                   # Step 4
    # gradapprox =                                # Step 5
    # YOUR CODE STARTS HERE
    theta_plus = theta + epsilon
    theta_minus = theta - epsilon
    J_plus = forward_propagation(x, theta_plus)
    J_minus = forward_propagation(x, theta_minus)
    gradapprox = (J_plus - J_minus) / (2 * epsilon)
    # YOUR CODE ENDS HERE

    # Check if gradapprox is close enough to the output of backward_propagation()
    # (approx. 1 line) DO NOT USE "grad = gradapprox"
    # grad =
    # YOUR CODE STARTS HERE
    grad = backward_propagation(x, theta)

    # YOUR CODE ENDS HERE

    # (approx. 3 lines)
    # numerator =                                 # Step 1'
    # denominator =                               # Step 2'
    # difference =                                # Step 3'
    # YOUR CODE STARTS HERE
    numerator = np.linalg.norm(grad - gradapprox)
    denominator = np.linalg.norm(grad) + np.linalg.norm(gradapprox)
    difference = numerator / denominator
    # YOUR CODE ENDS HERE
    if print_msg:
        if difference > 2e-7:
            print(
                "\033[93m"
                + "There is a mistake in the backward propagation! difference = "
                + str(difference)
                + "\033[0m"
            )
        else:
            print(
                "\033[92m"
                + "Your backward propagation works perfectly fine! difference = "
                + str(difference)
                + "\033[0m"
            )

    return difference

In [7]:
x, theta = 3, 4
difference = gradient_check(x, theta, print_msg=True)
gradient_check_test(gradient_check, difference)

Your backward propagation works perfectly fine! difference = 7.814075313343006e-11
 All tests passed.


In [ ]:
def forward_propagation_n(X, Y, parameters):
    """
    Implements the forward propagation (and computes the cost) presented in Figure 3.

    Arguments:
    X -- training set for m examples
    Y -- labels for m examples
    parameters -- python dictionary containing your parameters "W1", "b1", "W2", "b2", "W3", "b3":
                    W1 -- weight matrix of shape (5, 4)
                    b1 -- bias vector of shape (5, 1)
                    W2 -- weight matrix of shape (3, 5)
                    b2 -- bias vector of shape (3, 1)
                    W3 -- weight matrix of shape (1, 3)
                    b3 -- bias vector of shape (1, 1)

    Returns:
    cost -- the cost function (logistic cost for m examples)
    cache -- a tuple with the intermediate values (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3)

    """

    # retrieve parameters
    m = X.shape[1]
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]

    # LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)

    # Cost
    log_probs = np.multiply(-np.log(A3), Y) + np.multiply(-np.log(1 - A3), 1 - Y)
    cost = 1.0 / m * np.sum(log_probs)

    cache = (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3)

    return cost, cache

In [21]:
def backward_propagation_n(X, Y, cache):
    """
    Implement the backward propagation presented in figure 2.

    Arguments:
    X -- input datapoint, of shape (input size, 1)
    Y -- true "label"
    cache -- cache output from forward_propagation_n()

    Returns:
    gradients -- A dictionary with the gradients of the cost with respect to each parameter, activation and pre-activation variables.
    """

    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache

    dZ3 = A3 - Y
    dW3 = 1.0 / m * np.dot(dZ3, A2.T)
    db3 = 1.0 / m * np.sum(dZ3, axis=1, keepdims=True)

    dA2 = np.dot(W3.T, dZ3)
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    # dW2 = 1.0 / m * np.dot(dZ2, A1.T) * 2
    dW2 = 1.0 / m * np.dot(dZ2, A1.T)
    db2 = 1.0 / m * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1.0 / m * np.dot(dZ1, X.T)
    # db1 = 4.0 / m * np.sum(dZ1, axis=1, keepdims=True)
    db1 = 1.0 / m * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {
        "dZ3": dZ3,
        "dW3": dW3,
        "db3": db3,
        "dA2": dA2,
        "dZ2": dZ2,
        "dW2": dW2,
        "db2": db2,
        "dA1": dA1,
        "dZ1": dZ1,
        "dW1": dW1,
        "db1": db1,
    }

    return gradients

In [ ]:
# GRADED FUNCTION: gradient_check_n


def gradient_check_n(parameters, gradients, X, Y, epsilon=1e-7, print_msg=False):
    """
    Checks if backward_propagation_n computes correctly the gradient of the cost output by forward_propagation_n

    Arguments:
    parameters -- python dictionary containing your parameters "W1", "b1", "W2", "b2", "W3", "b3"
    grad -- output of backward_propagation_n, contains gradients of the cost with respect to the parameters
    X -- input datapoint, of shape (input size, number of examples)
    Y -- true "label"
    epsilon -- tiny shift to the input to compute approximated gradient with formula(1)

    Returns:
    difference -- difference (2) between the approximated gradient and the backward propagation gradient
    """

    # Set-up variables
    parameters_values, _ = dictionary_to_vector(parameters)

    grad = gradients_to_vector(gradients)
    num_parameters = parameters_values.shape[0]
    J_plus = np.zeros((num_parameters, 1))
    J_minus = np.zeros((num_parameters, 1))
    gradapprox = np.zeros((num_parameters, 1))

    # Compute gradapprox
    for i in range(num_parameters):

        # Compute J_plus[i]. Inputs: "parameters_values, epsilon". Output = "J_plus[i]".
        # "_" is used because the function you have outputs two parameters but we only care about the first one
        # (approx. 3 lines)
        # theta_plus =                                        # Step 1
        # theta_plus[i] =                                     # Step 2
        # J_plus[i], _ =                                     # Step 3
        # YOUR CODE STARTS HERE
        theta_plus = np.copy(parameters_values)
        theta_plus[i] = theta_plus[i] + epsilon
        J_plus[i], _ = forward_propagation_n(X, Y, vector_to_dictionary(theta_plus))
        # YOUR CODE ENDS HERE

        # Compute J_minus[i]. Inputs: "parameters_values, epsilon". Output = "J_minus[i]".
        # (approx. 3 lines)
        # theta_minus =                                    # Step 1
        # theta_minus[i] =                                 # Step 2
        # J_minus[i], _ =                                 # Step 3
        # YOUR CODE STARTS HERE
        theta_minus = np.copy(parameters_values)
        theta_minus[i] = theta_minus[i] - epsilon
        J_minus[i], _ = forward_propagation_n(X, Y, vector_to_dictionary(theta_minus))
        # YOUR CODE ENDS HERE

        # Compute gradapprox[i]
        # (approx. 1 line)
        # gradapprox[i] =
        # YOUR CODE STARTS HERE
        gradapprox[i] = (J_plus[i] - J_minus[i]) / (2 * epsilon)

        # YOUR CODE ENDS HERE

    # Compare gradapprox to backward propagation gradients by computing difference.
    # (approx. 3 line)
    # numerator =                                             # Step 1'
    # denominator =                                           # Step 2'
    # difference =                                            # Step 3'
    # YOUR CODE STARTS HERE
    numerator = np.linalg.norm(grad - gradapprox)
    denominator = np.linalg.norm(grad) + np.linalg.norm(gradapprox)
    difference = numerator / denominator
    # YOUR CODE ENDS HERE
    if print_msg:
        if difference > 2e-7:
            print(
                "\033[93m"
                + "There is a mistake in the backward propagation! difference = "
                + str(difference)
                + "\033[0m"
            )
        else:
            print(
                "\033[92m"
                + "Your backward propagation works perfectly fine! difference = "
                + str(difference)
                + "\033[0m"
            )

    return difference

In [16]:
X, Y, parameters = gradient_check_n_test_case()

cost, cache = forward_propagation_n(X, Y, parameters)
gradients = backward_propagation_n(X, Y, cache)
difference = gradient_check_n(parameters, gradients, X, Y, 1e-7, True)
expected_values = [0.2850931567761623, 1.1890913024229996e-07]
assert not (
    type(difference) == np.ndarray
), "You are not using np.linalg.norm for numerator or denominator"
assert (
    type(difference) == np.float64
), "The value must be a 64 bit floating point scalar."
assert np.any(
    np.isclose(difference, expected_values)
), "Wrong value. It is not one of the expected values"

There is a mistake in the backward propagation! difference = 0.2850931567761624


`gradient_check_n_debug`

<small>A diagnostic wrapper around gradient checking that pinpoints which parameters are wrong, not just whether something is wrong.</small>

---

The Core Loop Invariant

<small>

For each parameter index $i$, we perturb only $\theta[i]$ by $\pm\epsilon$, recompute $J$ both ways, and estimate the gradient numerically:

$$gradapprox[i] = \frac{J(\theta + \epsilon \cdot e_i) - J(\theta - \epsilon \cdot e_i)}{2\epsilon}$$

Everything else in $\theta$ stays frozen. This isolates the partial derivative with respect to exactly one parameter at a time. We run $2P$ forward passes total for $P$ parameters — expensive, which is why this is a debug tool only.

</small>

---

Element-wise Relative Difference

<small>

$$elementwise\_diff[i] = \frac{|grad[i] - gradapprox[i]|}{|grad[i]| + |gradapprox[i]| + 10^{-8}}$$

**Why not just subtract?** A raw difference of $0.001$ is catastrophic if the gradient is $0.000002$, but negligible if it's $10{,}000$. Consider two scenarios:

| Scenario | `grad[i]`  | `gradapprox[i]` | Absolute diff |
| :------- | :--------- | :-------------- | :------------ |
| 1        | $1000.001$ | $1000.000$      | $0.001$       |
| 2        | $0.000002$ | $0.000001$      | $0.000001$    |

Which one has a bigger bug? Intuitively, scenario 2 — the gradient is off by a factor of 2. But the absolute difference in scenario 2 is _smaller_.

Dividing by the scale of the numbers fixes this:

$$\text{rel\_diff}_i = \frac{|\text{grad}_i - \text{gradapprox}_i|}{|\text{grad}_i| + |\text{gradapprox}_i|}$$

Now scenario 1 gives $\approx \frac{0.001}{2000} = 5 \times 10^{-7}$ (fine), and scenario 2 gives $\frac{10^{-6}}{3 \times 10^{-6}} = 0.33$ (bug). Correct conclusion.

- Why $|grad| + |gradapprox|$ in the denominator? We don't know which one is correct — that's the whole point of the check. Their sum is a neutral, symmetric scale estimate that doesn't privilege either side. If both are correct they're nearly equal, so the sum $\approx 2 \times \text{true scale}$ — a perfectly valid normalizer.

- Why $+ 1e-8$? Guard against division by zero when both values are near zero. A tiny gradient that's slightly off shouldn't falsely scream "bug." The $10^{-8}$ is far below any meaningful gradient magnitude, so it has no effect otherwise.

</small>

---

Thresholds

<small>

- $< 10^{-7}$: Clean, backprop is correct.
- $10^{-5}$ to $10^{-7}$: Probably fine, worth a look.
- $> 10^{-3}$: Definite bug; check the flagged parameter.

</small>

---

What the `keys` list does

<small>

`keys[i]` maps each scalar index in the flat $\theta$ vector back to its parameter name ($W1$, $b1$, $W2$, ...). This is what lets the table print "W2 at index 35" instead of just a number — turning a numerical anomaly into a named, actionable diagnosis.

</small>

---

Limitations to Remember

<small>

- Does not work with dropout: Forward passes must be deterministic.
- Consistency check only: Verifies backprop is consistent with forward prop, not that forward prop itself is correct.
- Never run during training: Performs $O(2P)$ forward passes per check.

</small>


In [ ]:
def gradient_check_n_debug(parameters, gradients, X, Y, epsilon=1e-7, threshold=1e-4):
    """
    Diagnostic gradient checker that identifies which specific parameters
    have incorrect gradients in backward_propagation_n.

    Unlike gradient_check_n (which returns a single scalar difference),
    this function computes an element-wise relative difference across all
    parameters and prints a table of the worst offenders — mapping each
    anomalous index back to its named parameter (W1, b1, W2, etc.).

    How it works
    ------------
    For each parameter index i in the flattened θ vector:
      1. Perturb θ[i] by +ε → compute J_plus[i]
      2. Perturb θ[i] by -ε → compute J_minus[i]
      3. gradapprox[i] = (J_plus[i] - J_minus[i]) / (2ε)
      4. rel_diff[i] = |grad[i] - gradapprox[i]|
                       / (|grad[i]| + |gradapprox[i]| + 1e-8)

    The denominator uses the sum of absolute values (not the norm) to
    normalize per-element — measuring relative error at each position
    independently. The 1e-8 guards against division by zero when both
    values are near zero.

    Parameters
    ----------
    parameters : dict
        Network parameters {"W1", "b1", "W2", "b2", "W3", "b3"}.
    gradients : dict
        Output of backward_propagation_n — gradients w.r.t. all parameters.
    X : np.ndarray
        Input data, shape (n_features, m_examples).
    Y : np.ndarray
        True labels, shape (1, m_examples).
    epsilon : float, optional
        Perturbation size for finite difference. Default 1e-7.
        Smaller → more accurate approximation but more floating point noise.
    threshold : float, optional
        Minimum relative difference to flag in the output table. Default 1e-4.
        Entries below this are considered numerically clean.

    Returns
    -------
    None
        Prints a formatted table of flagged (index, parameter name, grad,
        gradapprox, rel_diff) entries. Empty table = backprop is correct.

    Limitations
    -----------
    - Does not work with dropout (forward passes must be deterministic).
    - Verifies consistency between backprop and forward prop only —
      a bug in forward prop will not be caught.
    - Runs 2P forward passes for P parameters. Use only for debugging,
      never during training.
    """

    parameters_values, keys = dictionary_to_vector(parameters)
    grad = gradients_to_vector(gradients)
    num_parameters = parameters_values.shape[0]

    J_plus = np.zeros((num_parameters, 1))
    J_minus = np.zeros((num_parameters, 1))
    gradapprox = np.zeros((num_parameters, 1))

    for i in range(num_parameters):
        theta_plus = np.copy(parameters_values)
        theta_plus[i] += epsilon
        J_plus[i], _ = forward_propagation_n(X, Y, vector_to_dictionary(theta_plus))

        theta_minus = np.copy(parameters_values)
        theta_minus[i] -= epsilon
        J_minus[i], _ = forward_propagation_n(X, Y, vector_to_dictionary(theta_minus))

        gradapprox[i] = (J_plus[i] - J_minus[i]) / (2 * epsilon)

    # Element-wise relative difference
    elementwise_diff = np.abs(grad - gradapprox) / (
        np.abs(grad) + np.abs(gradapprox) + 1e-8
    )

    # Print the worst offenders
    print(f"{'Index':<8} {'Key':<6} {'grad':>15} {'gradapprox':>15} {'rel_diff':>15}")
    print("-" * 65)
    for i in range(num_parameters):
        if elementwise_diff[i] > threshold:
            print(
                f"{i:<8} {keys[i]:<6} {grad[i,0]:>15.8f} {gradapprox[i,0]:>15.8f} {elementwise_diff[i,0]:>15.2e}"
            )


gradient_check_n_debug(parameters, gradients, X, Y)

Index    Key               grad      gradapprox        rel_diff
-----------------------------------------------------------------
20       b1          2.53163149      0.63290795        6.00e-01
21       b1          0.14900559      0.03725111        6.00e-01
22       b1         -0.25605205     -0.06401301        6.00e-01
23       b1          0.36182301      0.09045546        6.00e-01
35       W2          1.83160330      0.91580170        3.33e-01
36       W2          0.04903095      0.02451541        3.33e-01
37       W2         -0.21595908     -0.10797954        3.33e-01
38       W2          1.80563782      0.90281879        3.33e-01


In [22]:
# After fixing the backprop code
X, Y, parameters = gradient_check_n_test_case()

cost, cache = forward_propagation_n(X, Y, parameters)
gradients = backward_propagation_n(X, Y, cache)
difference = gradient_check_n(parameters, gradients, X, Y, 1e-7, True)
expected_values = [0.2850931567761623, 1.1890913024229996e-07]
assert not (
    type(difference) == np.ndarray
), "You are not using np.linalg.norm for numerator or denominator"
assert (
    type(difference) == np.float64
), "The value must be a 64 bit floating point scalar."
assert np.any(
    np.isclose(difference, expected_values)
), "Wrong value. It is not one of the expected values"
gradient_check_n_debug(parameters, gradients, X, Y)

Your backward propagation works perfectly fine! difference = 1.1890913023739721e-07
Index    Key               grad      gradapprox        rel_diff
-----------------------------------------------------------------
